# QA 06 — data regimes and edge cases

Numerical scale, noise, constant targets, overlap, and additional class counts.

> Run this notebook from top to bottom after installing the project with
> `pip install -e ".[notebooks]"`. Figures are genuine public-API outputs.
> Cells intentionally contain no assertions: automated invariants live in
> `tests/`, while this notebook is for human visual inspection.

In [ ]:
from pathlib import Path
import sys

candidate = Path.cwd().resolve()
while candidate != candidate.parent and not (candidate / "pyproject.toml").exists():
    candidate = candidate.parent
if not (candidate / "pyproject.toml").exists():
    raise RuntimeError("Open this notebook from inside the Mlektic repository.")
ROOT = candidate
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
from IPython.display import display
from sklearn.linear_model import LinearRegression, LogisticRegression
from mlektic import visualize_lr, visualize_logistic
from notebooks._support import case_heading, linear_case, multiclass_case

### `DATA-LR-NOISELESS`

**Inspect:** exact fit and near-zero endpoint metrics

In [ ]:
case_heading("DATA-LR-NOISELESS", "exact fit and near-zero endpoint metrics")
c=linear_case(2,noise=0.0)
display(visualize_lr(c.model,c.X,c.y,steps=8,max_frames=5,detail='complete'))

### `DATA-LR-SCALE`

**Inspect:** features with radically different numeric scales

In [ ]:
case_heading("DATA-LR-SCALE", "features with radically different numeric scales")
rng=np.random.default_rng(17)
X=np.column_stack([rng.normal(size=80),1000*rng.normal(size=80)])
y=1+2*X[:,0]-0.004*X[:,1]
m=LinearRegression().fit(X,y)
display(visualize_lr(m,X,y,steps=8,max_frames=5,detail='academic'))

### `DATA-LR-CONSTANT-TARGET`

**Inspect:** constant target without divide-by-zero visual artifacts

In [ ]:
case_heading("DATA-LR-CONSTANT-TARGET", "constant target without divide-by-zero visual artifacts")
X=np.linspace(-2,2,60).reshape(-1,1)
y=np.full(60,3.5)
m=LinearRegression().fit(X,y)
display(visualize_lr(m,X,y,steps=8,max_frames=5,detail='complete'))

### `DATA-LOG-OVERLAP`

**Inspect:** overlapping binary classes and calibrated probability curve

In [ ]:
case_heading("DATA-LOG-OVERLAP", "overlapping binary classes and calibrated probability curve")
rng=np.random.default_rng(19)
X=rng.normal(size=(100,1))
y=(X[:,0]+rng.normal(size=100)>0).astype(int)
m=LogisticRegression(max_iter=1000).fit(X,y)
display(visualize_logistic(m,X,y,steps=8,max_frames=5,show_loss=True))

### `DATA-MULTI-FOUR`

**Inspect:** four-class probability geometry and legend

In [ ]:
case_heading("DATA-MULTI-FOUR", "four-class probability geometry and legend")
c=multiclass_case(2,classes=4,string_labels=True)
display(visualize_logistic(c.model,c.X,c.y,steps=8,max_frames=5,class_focus='group-2',show_class_labels=True))